# Run this project's main sampler (`scalpel`)

One dataset, one seed, one config, one image encoder — the full budget sweep
for exactly that. The notebook ends in a single zip whose name states the
whole configuration, so an ablation (a different `IMAGE_ENCODER`, a different
`uncertainty_mode`, …) is a second run, not a second entry in a list (§2.0).

**`IMAGE_ENCODER` decides the ONE feature space every stage of the run uses**
— the coverage kernel, the disagreement probes inside `scalpel`, and the
final evaluation probe (`PLAN_IMPLEMENT.md` §6.2). `"conch"` reads an
already-built cache from `extract_vlm_features.ipynb`; it does not extract
one itself. `USE_TEXT` is the round-1 cold-start text prior — **not yet
implemented** (`sampling/scalpel/sampler.py` round 1 is plain MaxHerding,
`U=1`; see `CLAUDE.md`'s contribution #1) — so this notebook raises rather
than silently running without it.

**`USE_LORA`, `AUX_LOSS` and `AUGMENT` are the final-training axes**
(`PLAN_IMPLEMENT.md` §6.4/§6.5), run AFTER a budget's points are already
selected.

**Every final-training axis is final-training ONLY — none of them changes
what gets selected.** `AUGMENT` used to be the exception: it fed each round's
uncertainty probe freshly augmented pixels, and that made it change the
selected set (measured on histoset seed 42: only ~55% of the points overlapped
the un-augmented run, at every budget). The cause was unavoidable — the
provider can only augment the probe's *training* rows, since augmenting ~90k
pool rows per round costs ~27x, and the CellViT cell probe cannot be augmented
at all because its embeddings come from a cache with no pixels behind them. So
`disagreement = JS(visual, cell)` was comparing a noised probe against a clean
one and reading the noise as disagreement: JS came out 1.65x the frozen run's
on average (max 3.28x), which pushed sigma down faster and left the high
budgets *worse* balanced. `AUGMENT` now applies only to the final probe, so it
answers the question it is meant to answer. **Every budget in the sweep gets its own
final-training pass** (the confirmed full-curve choice: the research question
is whether LoRA helps more at low or high budgets, which a single
max-budget point cannot answer), so this pass costs roughly as many
extra forward/backward passes as there are budgets, on top of the sweep. The
default (`USE_LORA=False`, `AUX_LOSS="none"`, `AUGMENT="none"`) is the
frozen-embedding fast path unchanged — this is the control every
final-training run is compared against.

On a Kaggle **T4 x2** session the budget list is split across both GPUs, one
worker process per card. Set `PARALLEL = False` to force serial.

Per budget this writes selected indices with their per-step acquisition scores,
the probe weights, the test predictions, the metrics table, a
sanity report and a run log — see `main.py`. The last cell zips all of it.

Re-running the notebook skips a run whose `_results.pt` already exists, so a
session that hits the 12-hour limit can simply be run again.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
# The bound is REQUIRED here, not just in requirements.txt: `-U` upgrades to
# the newest release regardless of what the previous line just installed, and
# huggingface_hub 1.x makes `import transformers` (4.x, pinned <5 for CONCH's
# tokenizer) raise at import time -- "huggingface-hub>=0.34.0,<1.0 is
# required ... found 1.29.0". That killed a run at the very first import.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Silence Hugging Face progress bars for this session AND every spawned GPU
# worker (they inherit os.environ). Without this, `transformers` prints a
# per-parameter "Loading weights" bar on each from_pretrained, and under
# `tee_stdout` those land in the run log: measured at 2442 of 2528 lines
# (97%, 337 KB) in one 2-shard run, burying the 86 lines that said what the
# run was actually doing. Set as an env var rather than by calling
# `transformers.utils.logging.disable_progress_bar()`, because it must apply
# before the child process imports transformers and it works the same on
# whatever version the Kaggle image ships.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"
SEED = 42             # ONE seed. Sweeping seeds = re-running the notebook.

# This notebook always runs `scalpel` -- this project's own method. A
# baseline (random, margin, uncertainty_herding, ...) is run_al_baseline.ipynb's
# job; scalpel needs the CellViT cell view baselines never read.

# --- variant axes: pick ONE value per line for this run (§2.0) ---
IMAGE_ENCODER = "dinov2"        # dinov2 | conch
USE_TEXT = False                # True = round-1 cold-start text prior.
                                 # NOT YET IMPLEMENTED -- see the intro cell.
                                 # Leave False; the assert cell below enforces this.
# manual | conch_official | llm_short | llm_morphology
# Only read once USE_TEXT actually does something (see intro cell).
DESCRIPTION_STYLE = "llm_short"
CELL_POOLING = "mean"           # mean | rff | moments -- goes into OVERRIDES below

# --- SELECTION auxiliary loss -- runs DURING the AL loop, every round ---
#
# Added to the two probes' cross-entropy in ONE backward pass:
#
#   loss = CE(visual) + CE(cell) + SEL_AUX_WEIGHT * JS(visual, cell | confident)
#
# The JS term is computed on the UNLABELED pool (~22k rows the probes never
# otherwise see, against <=200 labeled ones), restricted to points where BOTH
# heads land in the top half of their own confidence. That mask is
# load-bearing, not a
# tuning knob: pool-wide JS(visual, cell) IS scalpel's per-point acquisition
# weight, so an unmasked term would minimise exactly what the sampler ranks
# on. Masking leaves the contested region -- the region scalpel wants to
# sample -- out of the loss.
#
# 0.0 disables it and reproduces the plain run bit-for-bit. Try 1 / 5 / 20.
# Check `pool_mask_fraction` in the trace: it says how much of the pool the
# term actually saw.
#
# Everything else about the term adapts to the dataset on its own, because
# the three are not the same shape: the confidence gate is a QUANTILE of each
# head's own distribution (an absolute 0.9 admitted 67% / 49% / 42% of
# pathmnist / histoset / skintissue -- 9 / 14 / 16 classes), and how many pool
# rows it uses is half the pool capped at 20k (the pools differ 4.6x). So one
# setting here is the same experiment on all three.
SEL_AUX_WEIGHT = 0.0            # 0 = off. Try 1 / 5 / 20.

# --- FINAL-TRAINING axes (§6.4/§6.5) -- run AFTER selection, per budget ---
# Note these are a DIFFERENT auxiliary loss from SEL_AUX_* above: AUX_LOSS
# acts on the encoder's features during the final pass (and needs USE_LORA,
# since a frozen encoder's features carry no gradient), while SEL_AUX_*
# acts on the two selection probes during acquisition. They are independent
# and can be combined.
USE_LORA = False                # True = fine-tune IMAGE_ENCODER's attention via LoRA
LORA_R = 8                      # LoRA rank; ignored when USE_LORA=False
# The ADAPTER's learning rate, separate from PROBE_LR (config.yaml, 1e-3).
# 1e-3 is right for a linear probe on frozen features and far too large for a
# rank-8 adapter: measured on real DINOv2 it moves the CLS feature 116.9% in
# L2 (cosine 0.318 with the frozen output) vs 82.9% / 0.652 at 1e-4, and a
# real histoset run at 1e-3 collapsed the encoder -- 2849 of 5600 test rows
# predicted as ONE class at budget 200, worst at low budgets (-2.8 sigma at
# 50, -3.8 at 75). Set it equal to PROBE_LR to reproduce that run.
LORA_LR = 1e-4                  # ignored when USE_LORA=False
AUX_LOSS = "none"               # none | center | supcon | triplet
AUGMENT = "none"                # none | flip_rotate

# CONCH is GATED on Hugging Face. Only needed when this run actually LOADS the
# CONCH checkpoint -- i.e. IMAGE_ENCODER="conch" AND the final-training pass is
# on (USE_LORA=True or AUGMENT != "none"). A plain CONCH run reads the feature
# cache extract_vlm_features.ipynb already published and needs no token at all.
# Leave "" to fall back to a Kaggle Secret named HF_TOKEN (the next cell wires
# it into the environment); the assert cell fails fast if neither is set.
HF_TOKEN = ""

# Axes worth sweeping for scalpel are in config/config.yaml, e.g.:
#   {"uncertainty_mode": "visual_margin"}  ablation: plain UHerding weight
#   {"missing_impute": "zero"}             patches with no nucleus
# ONE config per run. `{}` uses config.yaml as-is; an ablation is a SECOND
# run with its own OVERRIDES, which gets its own run_name and its own zip.
# CELL_POOLING above is folded in here rather than kept as a separate knob,
# since it is already a sampler_cfg key (`cell_pooling`).
OVERRIDES = {}
if CELL_POOLING != "mean":
    OVERRIDES = {**OVERRIDES, "cell_pooling": CELL_POOLING}
# The SELECTION auxiliary loss. Only sent when it is actually on, so a run
# with SEL_AUX_WEIGHT=0 produces exactly the config -- and exactly the
# run_name -- it produced before this axis existed.
if SEL_AUX_WEIGHT > 0.0:
    OVERRIDES = {
        **OVERRIDES,
        "pool_consistency_weight": SEL_AUX_WEIGHT,
    }

# Leave RUN_NAME None so each variant gets a collision-safe name of its own.
RUN_NAME = None

# Use both T4s when the session has them.
PARALLEL = True

# Split this run's budget list across both GPUs (non-prefix-exact samplers only).
SPLIT_BUDGETS = True

# Starting points only; the next cell searches for the real directories.
FEATURE_DIR = "/kaggle/input/datasets/nhtquyn/pathoactive"
CELLVIT_DIR = "/kaggle/input/datasets/cryandrrich/pathoactivenucleus"
# Only read when IMAGE_ENCODER="conch" -- the cache extract_vlm_features.ipynb
# published. Not needed (and not searched) for IMAGE_ENCODER="dinov2".
VLM_FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/vlm_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

# Where a .npz dataset is re-exported as memory-mappable .npy files.
#
# An AL run reads no pixels -- it opens the dataset for labels, sample IDs and
# the cache fingerprint -- but NPZDataset reads a .npz EAGERLY, and
# PathMNIST-224 is ~15 GiB of uint8. Two GPU workers each holding their own
# copy is ~30 GiB, which is the whole Kaggle session, so one is killed before
# the sampler starts. Mapping costs scratch disk once and nothing per worker.
# Ignored for ImageFolder datasets (histoset, skintissue), which read per file.
MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

In [ ]:
import os

from training.finetune import needs_pixels

# Does this run load an image ENCODER at all? Selection never does -- it reads
# the frozen feature cache -- so only the §6.4 final-training pass can need a
# checkpoint. `needs_pixels` is the single predicate main.py uses for the same
# decision, imported rather than restated so the two cannot drift.
LOADS_ENCODER = needs_pixels(USE_LORA, AUGMENT, AUX_LOSS)

if IMAGE_ENCODER == "dinov2":
    from huggingface_hub import snapshot_download

    # Downloaded whether or not the final-training pass runs: main.py extracts
    # DINOv2 features itself on a cache miss, so the checkpoint can be needed
    # even when LOADS_ENCODER is False.
    print("Downloading facebook/dinov2-base ...")
    snapshot_download(repo_id="facebook/dinov2-base")
elif not LOADS_ENCODER:
    # CONCH, frozen path: nothing to download. The features come from the
    # cache extract_vlm_features.ipynb published, and no token is needed.
    print("IMAGE_ENCODER='conch' with no final-training pass: reads the")
    print("published feature cache only -- no checkpoint, no HF token needed.")
else:
    # CONCH + final training: the checkpoint IS loaded, so the gated repo has
    # to be reachable. Resolve the token the same way extract_vlm_features.ipynb
    # does -- an explicit HF_TOKEN first, then a Kaggle Secret -- and put it in
    # the environment, which is where main.run picks it up (including inside
    # each spawned GPU worker, since those inherit os.environ).
    if not HF_TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient

            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
            print("[auth] using Kaggle Secret HF_TOKEN")
        except Exception as exc:
            print(f"[auth] no Kaggle Secret HF_TOKEN ({type(exc).__name__})")
    assert HF_TOKEN, (
        "IMAGE_ENCODER='conch' with USE_LORA/AUGMENT on loads the GATED CONCH "
        "checkpoint, which needs a Hugging Face token. Set HF_TOKEN in the EDIT "
        "cell, or add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets). "
        "Without one the download fails with a 401/403 partway into the run."
    )
    os.environ["HF_TOKEN"] = HF_TOKEN

    from huggingface_hub import login

    login(HF_TOKEN)
    # `conch` is not in requirements.txt -- only this path needs it, and it is
    # a heavy install (timm, transformers) that a frozen run should not pay.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/mahmoodlab/CONCH.git"])
    print("conch installed; CONCH checkpoint will load in the final-training pass")

In [ ]:
import yaml
import torch

import main
from sampling.specs import spec_for
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_cellvit_cache, find_data_root, find_visual_cache, find_vlm_cache
from utils.progress import format_duration

In [ ]:
DATA_ROOT = find_data_root()

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
# This notebook is the scalpel entry point -- baselines have their own
# (run_al_baseline.ipynb), so this is a constant, not a knob.
SAMPLER = "scalpel"

sampler_cfg = {**config.get("samplers", {}).get(SAMPLER, {}), **OVERRIDES}
spec = spec_for(SAMPLER)

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
# One configuration per run: a list here would produce several results under
# one archive name, which could then not say which result was which.
assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
assert isinstance(OVERRIDES, dict), "OVERRIDES is one config dict, not a list of variants"
assert SAMPLER == "scalpel", (
    "run_al_main.ipynb always runs scalpel; a baseline goes through "
    "run_al_baseline.ipynb instead"
)
assert IMAGE_ENCODER in ("dinov2", "conch"), f"Unknown IMAGE_ENCODER={IMAGE_ENCODER!r}"

# §6.1: a one-way relationship. USE_TEXT needs a text tower in the SAME space
# as the image tower, and DINOv2 has none -- but CONCH without USE_TEXT is a
# valid, useful run on its own (the "CONCH visual, coverage-only round 1"
# ablation), so the reverse is never asserted.
if USE_TEXT:
    assert IMAGE_ENCODER == "conch", (
        "USE_TEXT needs a text tower in the same space as the image tower; "
        "DINOv2 has none. Set IMAGE_ENCODER='conch'."
    )
    # Contribution #1 (CLAUDE.md) -- round 1 of sampling/scalpel/sampler.py is
    # still plain MaxHerding (U=1); no code path reads a text embedding yet.
    # Raising here beats the alternative of quietly running the round-1
    # ablation and reporting it as something it is not.
    raise NotImplementedError(
        "USE_TEXT=True: the round-1 cold-start text prior is not implemented "
        "yet (CLAUDE.md contribution #1 -- sampling/scalpel/sampler.py round 1 "
        "is plain MaxHerding and reads no text embedding). Leave USE_TEXT=False."
    )

assert AUX_LOSS in ("none", "center", "supcon", "triplet"), f"Unknown AUX_LOSS={AUX_LOSS!r}"
assert AUGMENT in ("none", "flip_rotate"), f"Unknown AUGMENT={AUGMENT!r}"
# §6.1: supcon/triplet need >=2 samples/class in a batch, or the loss would
# silently collapse toward ~0 for any class below that (training/losses.py
# raises the same condition again, per batch, as a second line of defense --
# this check catches it before any GPU time is spent).
# An auxiliary loss acts on the ENCODER's features, so it only does anything
# when the encoder is being trained. With USE_LORA=False the encoder is frozen
# and its features carry no gradient, making the aux term a constant that
# changes nothing -- a run that would be byte-identical to AUX_LOSS="none"
# while its filename says otherwise. main.py refuses it; assert here so the
# notebook fails in its config cell rather than minutes into the sweep.
if AUX_LOSS != "none":
    assert USE_LORA, (
        f"AUX_LOSS={AUX_LOSS!r} needs USE_LORA=True. The auxiliary losses act "
        "on encoder features, which have no gradient while the encoder is "
        "frozen, so the term would contribute nothing and the run would "
        "duplicate the AUX_LOSS='none' baseline under a different name."
    )
# NOTE: supcon/triplet used to require min(budget) >= 2*num_classes here. That
# is no longer the rule: those losses now DROP anchors with no same-class
# positive instead of raising, which is the standard formulation and is what
# makes them runnable at all on histoset (14 classes, smallest budget 25).

FINAL_TRAIN_CFG = {
    "use_lora": USE_LORA, "lora_r": LORA_R,
    # alpha = 2*r is the usual LoRA convention (the update is scaled by
    # alpha/r, so this keeps the effective scale at 2 whatever rank you pick).
    # Derived rather than configured so the two cannot drift apart.
    "lora_alpha": 2.0 * LORA_R,
    "lora_lr": LORA_LR,
    # 0.5 for every aux_loss kind; not a swept axis.
    "aux_loss": AUX_LOSS, "aux_weight": 0.5, "augment": AUGMENT,
}

# scalpel always needs the CellViT cell view.
if "cell_embeddings" in spec.needs:
    found = find_cellvit_cache(DATASET, SEED, hint=CELLVIT_DIR)
    assert found is not None, (
        f"No CellViT cache for {DATASET}_seed{SEED} under {CELLVIT_DIR} or "
        f"the default Kaggle input roots. Attach the extraction dataset."
    )
    CELLVIT_DIR = str(found)
    print("cellvit cache:", CELLVIT_DIR)
    # §6.3: crop_dino embeddings are a DINOv2 forward pass over each nucleus
    # crop; mixing them into a CONCH run would mix two incompatible image
    # spaces. main.py::_load_cell_view raises this too (it can only check
    # once the cache manifest is read); checked here as well so a CONCH run
    # fails before spending any GPU time, not partway through the sweep.
    assert not (IMAGE_ENCODER == "conch" and sampler_cfg.get("cell_source") == "crop_dino"), (
        "cell_source='crop_dino' was extracted with DINOv2; using it with "
        "IMAGE_ENCODER='conch' mixes two incompatible image spaces. Use "
        "cell_source='cellvit_embedding' instead."
    )

if IMAGE_ENCODER == "conch":
    # An AL run READS this cache; it never builds one. Building one needs the
    # conch package, an HF token and a slow 448x448 forward pass -- that is
    # extract_vlm_features.ipynb's job, run once and published as its own
    # Kaggle Dataset.
    vlm_name = config.get("models", {}).get("vlm", "MahmoodLab/CONCH")
    found = find_vlm_cache(DATASET, SEED, vlm_name, hint=VLM_FEATURE_DIR)
    assert found is not None, (
        f"No CONCH feature cache for {DATASET}_seed{SEED}_{vlm_name} under "
        f"{VLM_FEATURE_DIR} or the default Kaggle input roots. Run "
        f"extract_vlm_features.ipynb first and attach its published dataset."
    )
    VLM_FEATURE_DIR = str(found)
    print("vlm cache:", VLM_FEATURE_DIR)
else:
    # The DINOv2 cache is optional: main.py re-extracts on a miss. It must not
    # re-extract into a read-only /kaggle/input, which would only fail AFTER
    # the whole forward pass, so fall back to a writable directory instead.
    vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
    found = find_visual_cache(DATASET, SEED, vit_name, hint=FEATURE_DIR)
    if found is not None:
        FEATURE_DIR = str(found)
        print("features cache:", FEATURE_DIR)
    else:
        print(f"[features] not found for {DATASET}/seed{SEED}/{vit_name} - will extract this session.")
        print("  (a .npy with no matching manifest is rejected too -- row alignment")
        print("  cannot be verified without it.)")
        FEATURE_DIR = "/kaggle/working/features"
    if not str(FEATURE_DIR).startswith("/kaggle/input"):
        Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
print(f"image_encoder: {IMAGE_ENCODER}")
print(f"final_train_cfg: {FINAL_TRAIN_CFG}")
print(f"GPUs visible: {visible_gpu_count()}")
print("config:", sampler_cfg)

In [ ]:
import time

BUDGETS = config["cumulative_budget"]
workers = visible_gpu_count() if PARALLEL else 1

# Budget sharding only helps a sampler whose budgets are independent runs.
# `spec.prefix_exact` is exactly that property, so it -- not a hand-kept list
# of names -- decides.
shard_budgets = SPLIT_BUDGETS and workers > 1 and not spec.prefix_exact and len(BUDGETS) > 1
if SPLIT_BUDGETS and spec.prefix_exact:
    print(f"[shard] {SAMPLER} is prefix-exact: one shared selection pass covers every")
    print("        budget, so its sweep stays on a single GPU.")

def budget_shards(budgets, n):
    """Round-robin, not contiguous: cost grows with the budget, so a contiguous
    split would hand one worker every large budget."""
    groups = [budgets[i::n] for i in range(n)]
    return [g for g in groups if g]

# encoder=IMAGE_ENCODER so a DINOv2 run and a CONCH run of the same sampler
# config get distinct names, and final_train_cfg=FINAL_TRAIN_CFG so the §6.4
# axes (use_lora / aux_loss / augment) do too. Without the latter, a sweep
# over (cell_pooling x use_lora x aux_loss x augment) collapses 48 configs
# onto 3 names: each run overwrites the previous one's files, and the resume
# check below ("does <RUN>_results.pt exist") then SKIPS the other 45
# entirely, leaving a results table that looks complete but holds 3 configs.
RUN = RUN_NAME or main._default_run_name(
    SAMPLER, sampler_cfg, encoder=IMAGE_ENCODER, final_train_cfg=FINAL_TRAIN_CFG,
)
if SEED != config.get("random_seed", 42):
    RUN = f"{RUN}_s{SEED}"

base_kwargs = dict(
    data_path=str(data_path),
    sampler_name=SAMPLER,
    num_classes=dataset_info["num_classes"],
    data_descriptions=dataset_info.get("descriptions", {}),
    prompt_templates=config.get("prompt_templates", []),
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    random_seed=SEED,
    save_dir=str(SAVE_DIR),
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
    mmap_cache_dir=MMAP_CACHE_DIR,
    cellvit_cache_dir=CELLVIT_DIR,
    run_name=RUN,
    device_string="cuda:0",
    image_encoder=IMAGE_ENCODER,
    vlm_cache_dir=VLM_FEATURE_DIR,
    # Passed explicitly rather than left to the environment alone: a spawned
    # worker does inherit os.environ, but relying on that makes the token
    # invisible at this call site. None wherever the gated checkpoint is never
    # loaded -- main.run falls back to $HF_TOKEN in that case anyway.
    hf_token=(HF_TOKEN or None) if (IMAGE_ENCODER == "conch" and LOADS_ENCODER) else None,
    final_train_cfg=FINAL_TRAIN_CFG,
)

jobs = []
shard_tags = []
if (SAVE_DIR / f"{RUN}_results.pt").is_file():
    print(f"already finished, nothing to do: {RUN}_results.pt")
elif shard_budgets:
    shards = budget_shards(BUDGETS, workers)
    shard_tags = [f"shard{i}" for i in range(len(shards))]
    for tag, budgets in zip(shard_tags, shards):
        jobs.append((f"{RUN}:{tag}", dict(
            base_kwargs, cumulative_budget=budgets, shard_tag=tag,
        )))
else:
    jobs.append((RUN, dict(base_kwargs, cumulative_budget=BUDGETS)))

print(f"run: {RUN}")
for label, kwargs in jobs:
    print(f"   {label:40} budgets={kwargs['cumulative_budget']}")

# Export the .npz to memory-mappable .npy files ONCE, here in the parent.
#
# Must be the parent: two GPU workers exporting the same ~15 GiB concurrently
# would race on the same filenames. Once exported, each worker maps the same
# pages and the OS page cache serves them all from one copy, so a second worker
# costs no additional pixel memory.
#
# ImageFolder datasets skip this: they hold a path list and read per file, so
# they were never the problem.
if str(data_path).endswith(".npz"):
    from data.npz_mmap import export_npz_to_npy

    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

started = time.time()
results = run_variants_parallel(jobs, main.run_on_worker, num_workers=workers)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label']:40} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
if results:
    print(f"total {format_duration(time.time() - started)} | "
          f"{len(results) - len(failed)}/{len(results)} succeeded")
assert not failed, f"jobs failed: {failed}"

if shard_tags:
    linear = main.merge_budget_shards(str(SAVE_DIR), RUN, shard_tags)
    print(f"[merge] {RUN}: {len(linear)} budgets -> {RUN}_results.pt")

In [ ]:
# Results table, in budget order.
#
# The dispatch cell above runs two GPUs in one output stream, so their progress
# lines interleave and a budget's numbers can land anywhere. This cell ignores
# all of that and reads the saved `<run>_results.pt`, which is the authoritative
# record -- so the table below is correctly ordered no matter how the logs came
# out, and re-running this cell alone reprints it without recomputing anything.
#
# Metrics only. Per-step acquisition scores, sigma and the sanity report live in
# the saved files for later analysis; printing them here would recreate exactly
# the wall of text this cell exists to avoid.
import torch

results_path = SAVE_DIR / f"{RUN}_results.pt"
assert results_path.is_file(), (
    f"no results at {results_path} -- the run above did not finish"
)
payload = torch.load(results_path, weights_only=False)
linear = payload["linear"]

print(f"{payload['sampler']}  |  {payload['dataset']}  |  seed {payload['seed']}")
print(f"run_name: {payload['run_name']}")
if payload.get("sharded_over"):
    print(f"budget shards: {', '.join(payload['sharded_over'])}")
print()

header = f"{'budget':>8}  {'accuracy':>9}  {'precision':>9}  {'recall':>9}  {'macro F1':>9}  {'select s':>9}"
print(header)
print("-" * len(header))
for budget in sorted(linear):
    row = linear[budget]
    print(f"{budget:>8}  {row['acc']:>9.4f}  {row['precision']:>9.4f}  "
          f"{row['recall']:>9.4f}  {row['f1']:>9.4f}  {row['selection_seconds']:>9.1f}")
print("-" * len(header))

best = max(linear, key=lambda b: linear[b]["acc"])
print(f"best accuracy {linear[best]['acc']:.4f} at budget {best}")

# A sanity severity worse than "ok" means the selection itself looked
# degenerate at some budget -- worth seeing next to the numbers rather than
# buried in the log above.
worst = {b: linear[b].get("sanity_severity", "ok") for b in sorted(linear)}
flagged = {b: s for b, s in worst.items() if s != "ok"}
if flagged:
    print(f"sanity: {flagged}  <- check the run log for details")
else:
    print("sanity: ok at every budget")

In [ ]:
# Package the results as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape both extraction notebooks and
# run_al_baseline.ipynb use.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI. Keeping the originals beside the
# zip also doubles the download, and a session over the ~20 GB Output quota
# shows NOTHING at all, including the files that were fine.
import shutil

from utils import main_archive_stem

# Delete the .npy mmap export BEFORE archiving.
#
# It is scratch for the pixel-reading pass only -- nothing below it, and
# nothing downstream, ever reads it again. But it is the single largest thing
# in /kaggle/working: PathMNIST-224 exports ~15 GiB, and a real run of this
# notebook finished with 16.1 GB sitting in Output, ~15.7 GB of it this
# directory. That is 80% of the ~20 GB Output quota spent on a temporary
# file, and a session that goes over the quota shows NOTHING in the Output
# tab -- including the zip that was fine. Same failure the nucleus run hit
# (CLAUDE.md: "The nucleus run dies of DISK, not memory").
if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")
SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
# make_archive must not write inside the directory being archived, or it packs
# a partial copy of itself on a second run.
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

# encoder=IMAGE_ENCODER so a DINOv2 run and a CONCH run of the same sampler
# config get distinct archive names too, not just distinct run_names --
# otherwise the second publish would overwrite the first's zip on Kaggle
# Datasets (PLAN_IMPLEMENT.md §6.5, the archive-layer version of §6.2).
# run_name=RUN so each of the 48 sweep configurations gets its OWN zip.
# Without it every one of them is `pathmnist_scalpel_seed42.zip`: each
# download overwrites the last, and each Kaggle Dataset publish versions
# over the previous configuration instead of sitting beside it.
STEM = main_archive_stem(DATASET, SAMPLER, SEED, encoder=IMAGE_ENCODER, run_name=RUN)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
  3. In evaluate_al_sampler.ipynb: Add Data -> your new dataset, then point
     CHECKPOINT_ROOT at it to rebuild the table and fit PALM.""")